# Capacities, the Choquet integral, and interpretation

A finite capacity assigns importance to coalitions of variables. It is normalized and monotone. The discrete Choquet integral aggregates normalized inputs using those coalition values; the additive case is a weighted mean.


In [7]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from capacities_ml.capacities import (
    ExplicitCapacity,
    KAdditiveCapacity,
    VariableUniverse,
    inverse_mobius_transform,
    mobius_transform,
)
from capacities_ml.integrals.batch_integrals import (
    batch_choquet_integral,
    batch_choquet_integral_mobius,
)
from capacities_ml.integrals.choquet import mobius_choquet, ordered_choquet
from capacities_ml.interpretation import (
    interaction_signs,
    pairwise_interaction_matrix,
    pairwise_interactions,
    shapley_indices,
)

np.set_printoptions(precision=4, suppress=True)


## 1. Define a finite capacity

The values below satisfy `nu(empty set) = 0`, `nu(N) = 1`, and monotonicity. Coalition values need not equal the sum of singleton values.


In [8]:
universe = VariableUniverse(("profitability", "liquidity", "solvency"))

capacity = ExplicitCapacity(
    universe=universe,
    values={
        "profitability": 0.25,
        "liquidity": 0.20,
        "solvency": 0.15,
        ("profitability", "liquidity"): 0.70,
        ("profitability", "solvency"): 0.60,
        ("liquidity", "solvency"): 0.30,
        ("profitability", "liquidity", "solvency"): 1.00,
    },
)

pd.Series(capacity.to_named_dict(), name="capacity value")


nan            nan        nan         0.00
profitability  nan        nan         0.25
liquidity      nan        nan         0.20
solvency       nan        nan         0.15
profitability  liquidity  nan         0.70
               solvency   nan         0.60
liquidity      solvency   nan         0.30
profitability  liquidity  solvency    1.00
Name: capacity value, dtype: float64

## 2. Build the equivalent 2-additive object

A 2-additive capacity keeps singleton and pairwise Möbius terms. Higher-order coalition values are completed automatically.


In [9]:
capacity_2add = KAdditiveCapacity(
    universe=universe,
    values={
        "profitability": 0.25,
        "liquidity": 0.20,
        "solvency": 0.15,
        ("profitability", "liquidity"): 0.70,
        ("profitability", "solvency"): 0.60,
        ("liquidity", "solvency"): 0.30,
    },
    k=2,
)

pd.Series(capacity_2add.to_named_dict(), name="2-additive capacity")


nan            nan        nan         0.00
profitability  nan        nan         0.25
liquidity      nan        nan         0.20
solvency       nan        nan         0.15
profitability  liquidity  nan         0.70
               solvency   nan         0.60
liquidity      solvency   nan         0.30
profitability  liquidity  solvency    1.00
Name: 2-additive capacity, dtype: float64

## 3. Möbius representation

Singleton coefficients describe individual effects. Pair coefficients capture complementarity when positive and redundancy when negative. The inverse transform recovers the original capacity.


In [10]:
mobius = mobius_transform(capacity)
recovered_capacity = inverse_mobius_transform(mobius)

print("Möbius coefficients")
display(pd.Series(mobius.to_named_dict(), name="m(A)"))
original_values = capacity.to_named_dict()
recovered_values = recovered_capacity.to_named_dict()
same_values = np.allclose(
    list(original_values.values()),
    [recovered_values[coalition] for coalition in original_values],
)
print("Recovered capacity equals the original:", same_values)


Möbius coefficients


profitability  nan        nan         0.25
liquidity      nan        nan         0.20
solvency       nan        nan         0.15
profitability  liquidity  nan         0.25
               solvency   nan         0.20
liquidity      solvency   nan        -0.05
profitability  liquidity  solvency    0.00
Name: m(A), dtype: float64

Recovered capacity equals the original: True


## 4. One observation and a batch

The ordered and Möbius formulas are equivalent. Batch functions apply the same aggregation row by row.


In [11]:
x = np.array([0.70, 0.40, 0.80])
X = np.array(
    [
        [0.70, 0.40, 0.80],
        [0.30, 0.90, 0.50],
        [0.80, 0.75, 0.60],
        [0.25, 0.35, 0.20],
    ]
)

print("Ordered formula:", ordered_choquet(capacity, x))
print("Möbius formula:", mobius_choquet(mobius, x))
print("Batch from capacity:", batch_choquet_integral(X, capacity))
print("Batch from Möbius coefficients:", batch_choquet_integral_mobius(X, mobius))


Ordered formula: 0.595
Möbius formula: 0.595
Batch from capacity: [0.595  0.44   0.7175 0.255 ]
Batch from Möbius coefficients: [0.595  0.44   0.7175 0.255 ]


## 5. Interpretation

Shapley indices measure average marginal importance. Pairwise indices measure whether variables reinforce or substitute for each other.


In [12]:
shapley = shapley_indices(capacity)
interactions = pairwise_interactions(capacity)
signs = interaction_signs(capacity)
interaction_matrix = pairwise_interaction_matrix(capacity)

print("Shapley importance")
display(pd.Series(shapley, name="Shapley index"))
print("Pairwise interactions")
display(pd.DataFrame({"interaction": interactions, "sign": signs}))
print("Interaction matrix")
display(pd.DataFrame(interaction_matrix, index=universe.var_names, columns=universe.var_names))


Shapley importance


profitability    0.475
liquidity        0.300
solvency         0.225
Name: Shapley index, dtype: float64

Pairwise interactions


interaction  sign
profitability liquidity         0.25     1
              solvency          0.20     1
liquidity     solvency         -0.05    -1

Interaction matrix


,profitability,liquidity,solvency
profitability,0.00,0.25,0.20
liquidity,0.25,0.00,-0.05
solvency,0.20,-0.05,0.00
